# Lubbock district crash extract

This notebook is the pass over `district_crashes.csv` before the API. It keeps to the fields the application uses: what each one is for, what is wrong with it, and the crash-level figures those decisions produce.

The file is a TxDOT CRIS export for the Lubbock district, injury and fatal crashes only, years 2020 through 2026. The preamble says the extract was current as of 22 October 2026.


## The file is not a plain table

The first records are a disclaimer, the query size, and the filters TxDOT applied. The column names begin on the row whose first field is `Crash ID`. A reader that treats line 1 as the header will not see the crash table.


In [1]:
from pathlib import Path

ROOT = Path.cwd()
if not (ROOT / "district_crashes.csv").exists():
    ROOT = ROOT.parent
CSV_PATH = ROOT / "district_crashes.csv"
if not CSV_PATH.exists():
    raise FileNotFoundError("Run this notebook from the project root.")

header_line = None
preamble = []
with CSV_PATH.open(encoding="utf-8-sig") as handle:
    for index, line in enumerate(handle):
        if line.startswith('"Crash ID"') or line.startswith("Crash ID"):
            header_line = index
            break
        text = line.strip()
        if text:
            preamble.append(text)

print(f"Header is physical line {header_line + 1}.")
print("Preamble:")
for line in preamble:
    print(line[:220])


Header is physical line 12.
Preamble:
"All crash data available using this tool represents reportable data collected from Texas Peace Officer's Crash Reports (CR-3) received and processed by the Texas Department of Transportation (Department) as of 10/22/202
"Query Result Counts:"
"Your query returned a total of 24,261 Crashes containing 48,043 Units and 67,906 Persons"
"Filters Applied to current Query:"
"Crash Year Is In  2020 or 2021 or 2022 or 2023 or 2024 or 2025 or 2026"
"TxDOT District Is Equal To LUBBOCK"
"Crash Severity Is In A - SUSPECTED SERIOUS INJURY or B - SUSPECTED MINOR INJURY or K - FATAL INJURY or C - POSSIBLE INJURY"


## Columns this project keeps

Ninety-four columns come back from the query. These are the ones the counts and the map use.

| Column | Why it is here |
|---|---|
| `Crash ID` | Identity. Person rows that share an id are the same crash. |
| `Crash Severity` | The K / A / B / C injury level. |
| `Contributing Factors` | Every coded factor for the crash, joined with semicolons. |
| `Contributing Factor 1` | Checked against the combined field, then not used for the rank. |
| `County` | Filter. The district file covers more than Lubbock County. |
| `Crash Date`, `Crash Year`, `Crash Time` | Month, year, and hour. |
| `Road Class` | Filter. |
| `Latitude`, `Longitude` | The map. Missing and broken points stay in the counts. |
| `Crash Death Count` | Deaths on the crash, which is not the same number as fatal crashes. |

`Contributing Factor 2` and `Contributing Factor 3` repeat the same pattern as factor 1, with still more missing values, so they are not profiled separately.


## One crash is stored once per person

TxDOT's own preamble says the query returned 24,261 crashes and 67,906 persons. The table matches that: each data row is a person, and the crash attributes are copied onto every person in that crash. Severity counts taken from the raw rows are person counts.


In [2]:
import pandas as pd

COLUMNS = [
    "Crash ID",
    "Crash Severity",
    "Contributing Factors",
    "Contributing Factor 1",
    "County",
    "Crash Date",
    "Crash Year",
    "Crash Time",
    "Latitude",
    "Longitude",
    "Road Class",
    "Crash Death Count",
]

# keep_default_na is off so the literal values "NA" and "None" are not
# silently turned into nulls. This extract mostly says "No Data" and "NONE".
raw = pd.read_csv(
    CSV_PATH,
    skiprows=header_line,
    usecols=COLUMNS,
    dtype=str,
    keep_default_na=False,
)
crash = raw.drop_duplicates("Crash ID")
rows_per_crash = raw.groupby("Crash ID").size()

print(f"Person rows: {len(raw):,}")
print(f"Crash ids:   {raw['Crash ID'].nunique():,}")
print("Rows per crash:")
print(rows_per_crash.describe().round(2).to_string())
print()
print("Severity disagreements within a crash:", int((raw.groupby("Crash ID")["Crash Severity"].nunique() > 1).sum()))
print("Factor disagreements within a crash:", int((raw.groupby("Crash ID")["Contributing Factors"].nunique() > 1).sum()))
print()
print("Severity, counting person rows:")
print(raw["Crash Severity"].value_counts().to_string())
print()
print("Severity, counting crashes:")
print(crash["Crash Severity"].value_counts().to_string())
print()
death_rows = pd.to_numeric(raw["Crash Death Count"], errors="coerce").fillna(0)
death_crashes = pd.to_numeric(crash["Crash Death Count"], errors="coerce").fillna(0)
print(f"Death count summed on person rows: {int(death_rows.sum()):,}")
print(f"Death count summed on crashes:     {int(death_crashes.sum()):,}")


Person rows: 67,906
Crash ids:   24,261
Rows per crash:
count    24261.00
mean         2.80
std          1.73
min          1.00
25%          2.00
50%          2.00
75%          3.00
max         58.00



Severity disagreements within a crash: 0
Factor disagreements within a crash: 0

Severity, counting person rows:


Crash Severity
C - POSSIBLE INJURY             43664
B - SUSPECTED MINOR INJURY      18793
A - SUSPECTED SERIOUS INJURY     3842
K - FATAL INJURY                 1607

Severity, counting crashes:
Crash Severity
C - POSSIBLE INJURY             14867
B - SUSPECTED MINOR INJURY       7224
A - SUSPECTED SERIOUS INJURY     1546
K - FATAL INJURY                  624

Death count summed on person rows: 1,953
Death count summed on crashes:     707


The severity label and the combined factor text do not change from one person row to the next inside a crash. Keeping the first row for each `Crash ID` does not throw away a second, different crash story. It does change the totals: the person-level table has 1,607 rows marked fatal, and the crash-level table has 624 fatal crashes. Summing `Crash Death Count` on the raw rows multiplies deaths the same way.

The first row is the rule in `app/cleaning.py`.


## Missing values, and the word NONE

Blank fields are not the only empty value. TxDOT also writes `No Data`, `N/A`, and `NONE`. `NONE` on a contributing factor means the report did not list a factor. That is a real code, and it would dominate a frequency rank, so the rank leaves it out. It is not treated as a crash that failed to load.


In [3]:
SENTINELS = {"", "NO DATA", "N/A", "NA", "NONE", "NOT REPORTED", "NOT APPLICABLE", "UNKNOWN"}

print("Crash-level rows whose value is a sentinel:")
for column in COLUMNS:
    if column == "Crash ID":
        continue
    values = crash[column].str.strip().str.upper()
    print(f"  {column:<28} {int(values.isin(SENTINELS).sum()):>6,}")

factors = crash["Contributing Factors"].str.strip().str.upper()
print()
print("Crashes whose combined factor text is exactly NONE:", int((factors == "NONE").sum()))
print("Crashes whose combined factor text is blank or No Data:", int(factors.isin(["", "NO DATA", "N/A"]).sum()))


Crash-level rows whose value is a sentinel:
  Crash Severity                    0


  Contributing Factors            104
  Contributing Factor 1         1,363
  County                            0


  Crash Date                        0
  Crash Year                        0
  Crash Time                        0


  Latitude                        849
  Longitude                       849
  Road Class                        0


  Crash Death Count                 0

Crashes whose combined factor text is exactly NONE: 104
Crashes whose combined factor text is blank or No Data: 0


## Which factor column to rank

`Contributing Factor 1` prefixes the same words with a numeric code, and it is often `No Data` on crashes where `Contributing Factors` is filled in. The combined field is the one that actually lists every factor on the crash. Factors are separated with semicolons, and the same factor is sometimes repeated, so the loader splits, trims, uppercases, and keeps each label once per crash.


In [4]:
print(crash[["Contributing Factors", "Contributing Factor 1"]].head(6).to_string(index=False))
print()
combined = crash["Contributing Factors"].str.strip()
coded = crash["Contributing Factor 1"].str.strip()
useful = ~combined.str.upper().isin(["", "NO DATA", "NONE", "N/A", "NA"])
coded_missing = coded.str.upper().isin(["", "NO DATA", "N/A", "NA"])
print(
    f"Crashes with a usable combined factor but no Factor 1 value: "
    f"{int((useful & coded_missing).sum()):,} of {int(useful.sum()):,}"
)


                                    Contributing Factors                             Contributing Factor 1
                            DISREGARD STOP SIGN OR LIGHT                 16 - DISREGARD STOP SIGN OR LIGHT
            FAILED TO YIELD RIGHT OF WAY - PRIVATE DRIVE 34 - FAILED TO YIELD RIGHT OF WAY - PRIVATE DRIVE
                                            UNSAFE SPEED                                 60 - UNSAFE SPEED
                            OTHER (EXPLAIN IN NARRATIVE)                 98 - OTHER (EXPLAIN IN NARRATIVE)
FAILED TO CONTROL SPEED;CELL/MOBILE DEVICE USE - TEXTING                      22 - FAILED TO CONTROL SPEED
                                                    NONE                                           No Data

Crashes with a usable combined factor but no Factor 1 value: 1,259 of 24,157


## Cleaning rules

`load_export` in `app/cleaning.py` is the implementation of the decisions above.

- Skip every record until the `Crash ID` header, then read the table with the CSV parser so commas inside the disclaimer stay inside one field.
- Keep the first row for each crash id.
- Read severity from the leading `K`, `A`, `B`, or `C`.
- Split `Contributing Factors` on semicolons. Drop blanks, `NONE`, `NO DATA`, `N/A`, `UNKNOWN`, and `NOT APPLICABLE`. A crash can still contribute to more than one remaining factor.
- Take year and month from `Crash Date`. Take the hour from `Crash Time`, which is stored as HHMM. `Crash Year` is only a fallback when the date is missing.
- Store county and road class in uppercase.
- Keep a latitude and longitude only when both parse and both fall inside a broad Texas window (latitude 25 to 37, longitude -107 to -93). Other crashes remain in every count and are left off the map.
- Treat a non-numeric or negative death count as zero. Sum deaths after the crash-level collapse.

`OTHER (EXPLAIN IN NARRATIVE)` is kept. It is a coded factor. It does not need to be deleted to make the top five readable: on this extract it ranks seventh.


## Figures the API serves

The block below calls the same loader the processing script uses, so the notebook and the database are not two different interpretations of the file.


In [5]:
import sys
from collections import Counter

sys.path.insert(0, str(ROOT))
from app.cleaning import SEVERITY_LABELS, SEVERITY_ORDER, load_export

export = load_export(CSV_PATH)
records = export.crashes
severity = Counter(record.severity for record in records)
factors = Counter(factor for record in records for factor in record.factors)
years = Counter(record.crash_year for record in records)
counties = Counter(record.county for record in records)
roads = Counter(record.road_class for record in records)
hours = Counter(record.crash_hour for record in records)

assert export.district == "LUBBOCK"
assert export.source_as_of == "2026-10-22"
assert export.person_rows == 67_906
assert len(records) == 24_261
assert dict(severity) == {"K": 624, "A": 1546, "B": 7224, "C": 14867}
assert factors.most_common(5) == [
    ("FAILED TO CONTROL SPEED", 5559),
    ("FAILED TO YIELD RIGHT OF WAY - STOP SIGN", 2428),
    ("FAILED TO YIELD RIGHT OF WAY - TURNING LEFT", 2346),
    ("FAILED TO DRIVE IN SINGLE LANE", 2334),
    ("UNSAFE SPEED", 2034),
]

deaths = sum(record.death_count for record in records)
deaths_on_fatal = sum(record.death_count for record in records if record.severity == "K")
multi_death = sum(1 for record in records if record.death_count > 1)
highest_death_count = max(record.death_count for record in records)
unmapped = sum(1 for record in records if record.latitude is None)
no_factor = sum(1 for record in records if not record.factors)
latest = max(record.crash_date for record in records)
peak_hour, peak_hour_count = hours.most_common(1)[0]

print(f"District {export.district}, extract as of {export.source_as_of}.")
print(f"{export.person_rows:,} person rows -> {len(records):,} crashes.")
print()
print("Severity")
for code in SEVERITY_ORDER:
    print(f"  {code}  {SEVERITY_LABELS[code]:<28} {severity[code]:>6,}")
print(
    f"Deaths: {deaths:,}, all of them on K crashes. "
    f"{multi_death:,} crashes killed more than one person "
    f"(highest death count {highest_death_count})."
)
print(f"Latest crash date: {latest}.")
print(f"No usable coordinate: {unmapped:,}. No coded factor after cleaning: {no_factor:,}.")
print(f"Busiest hour: {peak_hour:02d}:00 ({peak_hour_count:,} crashes).")
print()
print("Crashes by year")
for year in sorted(years):
    print(f"  {year}  {years[year]:>6,}")
print()
print("Top counties")
for name, count in counties.most_common(5):
    print(f"  {name:<12} {count:>6,}")
print(f"  {len(counties)} counties in the district file.")
print()
print("Road class")
for name, count in roads.most_common():
    print(f"  {name:<24} {count:>6,}")
print()
print("Top five contributing factors")
for rank, (name, count) in enumerate(factors.most_common(5), start=1):
    print(f"  {rank}. {name} ({count:,})")
assert deaths == deaths_on_fatal == 707
assert multi_death == 66
assert highest_death_count == 5


District LUBBOCK, extract as of 2026-10-22.
67,906 person rows -> 24,261 crashes.

Severity
  K  Fatal injury                    624
  A  Suspected serious injury      1,546
  B  Suspected minor injury        7,224
  C  Possible injury              14,867
Deaths: 707, all of them on K crashes. 66 crashes killed more than one person (highest death count 5).
Latest crash date: 2026-10-20.
No usable coordinate: 849. No coded factor after cleaning: 104.
Busiest hour: 17:00 (2,171 crashes).

Crashes by year
  2020   3,351
  2021   3,971
  2022   3,716
  2023   3,673
  2024   3,452
  2025   3,444
  2026   2,654

Top counties
  LUBBOCK      19,100
  HALE            830
  GAINES          739
  HOCKLEY         638
  DAWSON          443
  17 counties in the district file.

Road class
  CITY STREET              10,677
  US & STATE HIGHWAYS       9,296
  FARM TO MARKET            1,406
  INTERSTATE                1,358
  NON TRAFFICWAY              782
  COUNTY ROAD                 721
  OTHER ROA

## Reading the figures

There are 24,261 injury or fatal crashes in the extract, not 67,906. Of those, 624 were fatal and 707 people died. Every recorded death sits on a K crash. The gap between 624 and 707 is multiple fatalities: 66 crashes killed more than one person, and the highest death count on one crash is 5.

Failure to control speed is the modal coded factor, on 5,559 crashes. The next three are a stop-sign yield, a left-turn yield, and failure to drive in a single lane. `OTHER (EXPLAIN IN NARRATIVE)` is seventh, with 1,604 crashes, so leaving it in the dictionary does not crowd the top five.

Lubbock County is 19,100 of the crashes, about 79 percent. The other 16 counties in the district file are the reason county is a filter rather than a label on the whole extract. City streets are the largest road class, then US and state highways.

The series runs from 1 January 2020 through 20 October 2026. The 2026 total of 2,654 crashes is a partial year ending two days before the extract date. It is not a finished year-over-year decline. The afternoon peak is 17:00, with 2,171 crashes, and every crash in this extract had a usable time.

849 crashes have no usable latitude or longitude. On this extract those values are sentinels such as No Data; every numeric coordinate fell inside the Texas window, so that check did not remove any further crash from the map. 104 crashes have no coded factor after cleaning, and each of them is a crash whose factor text was exactly NONE.
